# Phase 1: Exploratory Data Analysis (EDA) & ETL Pipeline
**Project:** Predictive Modeling of Cart Abandonment in Quick-Commerce Networks (AMLBT)
**Objective:** Perform rigorous data extraction, exploratory analysis, feature engineering, and transformation to prepare the raw supply chain data for machine learning models.

---
## 1. Environment Setup & Data Extraction (Load)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings

# Styling for professional academic visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("muted")
warnings.filterwarnings('ignore')

# Load the raw dataset
try:
    df = pd.read_csv('inventory_optimization.csv')
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print("Error: 'inventory_optimization.csv' not found in the root directory.")

display(df.head())

In [ ]:
# Initial Data Audit
print("--- Data Types and Missing Values ---")
display(df.info())

print("\n--- Statistical Summary ---")
display(df.describe().T)

## 2. Exploratory Data Analysis (EDA)
Analyzing the target variable distribution and feature correlations to understand supply chain friction.

In [ ]:
# Dynamically identify target variable
target_col = 'stockout_occurred'
if target_col not in df.columns:
    target_col = df.columns[-1]  # Fallback to last column

# Target Variable Distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x=target_col)
plt.title('Distribution of Target Variable (Stockout Events)')
plt.xlabel('Stockout Occurred (0 = No, 1 = Yes)')
plt.ylabel('Frequency')

# Add percentage labels
total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2 - 0.05
    y = p.get_y() + p.get_height() + (total * 0.01)
    ax.annotate(percentage, (x, y), ha='center')

plt.show()

In [ ]:
# Correlation Heatmap to identify multicollinearity and predictive strength
plt.figure(figsize=(10, 8))
corr_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 3. Data Transformation & Feature Engineering (ETL)
Cleaning anomalies, handling nulls, and engineering domain-specific metrics (e.g., Inventory Deficit Risk).

In [ ]:
df_clean = df.copy()

# 1. Drop Duplicate Rows
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Dropped {initial_rows - len(df_clean)} duplicate rows.")

# 2. Handle Missing Values
# Drop rows where the target is missing
df_clean = df_clean.dropna(subset=[target_col])

# Impute numerical missing values with the median (robust to outliers)
num_features = df_clean.select_dtypes(include=[np.number]).columns.tolist()
num_features.remove(target_col)

for col in num_features:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        
print("Missing values handled successfully.")

In [ ]:
# 3. Feature Engineering: Creating composite supply chain metrics
print("Engineering new features...")

try:
    # Example: Lead Time Demand = Lead Time * Daily Demand
    # Note: Replace 'lead_time' and 'demand' with exact column names from your data_dictionary_inventory.csv if different.
    demand_col = 'demand' if 'demand' in df_clean.columns else df_clean.columns[2]
    lead_time_col = 'lead_time' if 'lead_time' in df_clean.columns else df_clean.columns[1]
    inventory_col = 'inventory_level' if 'inventory_level' in df_clean.columns else df_clean.columns[3]

    df_clean['lead_time_demand_proxy'] = df_clean[lead_time_col] * df_clean[demand_col]
    
    # Example: Safety Stock Margin = On Hand Inventory - Lead Time Demand Proxy
    df_clean['safety_stock_margin'] = df_clean[inventory_col] - df_clean['lead_time_demand_proxy']
    
    print("New features 'lead_time_demand_proxy' and 'safety_stock_margin' created.")
except Exception as e:
    print(f"Could not engineer specific composite features due to column naming: {e}. Proceeding with base features.")

# Update numerical features list to include engineered features
num_features = df_clean.select_dtypes(include=[np.number]).columns.tolist()
num_features.remove(target_col)

In [ ]:
# 4. Feature Scaling / Normalization
# Tree-based models (XGBoost/RF) don't strictly require scaling, 
# but our Logistic Regression baseline requires Standardized inputs.

scaler = StandardScaler()
df_clean[num_features] = scaler.fit_transform(df_clean[num_features])

print("Continuous features standardized.")
display(df_clean.head())

## 4. Export Cleaned Dataset for Model Training
Saving the transformed DataFrame to be ingested by `QCommerce_Stockout_Model.ipynb`.

In [ ]:
# Save the final processed dataset
output_file = 'cleaned_inventory_optimization.csv'
df_clean.to_csv(output_file, index=False)

print(f"ETL Pipeline Complete! Cleaned dataset saved as '{output_file}'.")
print(f"Final Dataset Shape: {df_clean.shape}")